In [ ]:
# Repository setup for portable, repo-relative paths
from pathlib import Path
import sys

def _find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from dx_chat_entropy.paths import get_paths
PATHS = get_paths(REPO_ROOT)


# Differential LR Estimation

Estimate pairwise **differential likelihood ratios** for clinical findings
that discriminate between two diagnosis categories (`dx_cat_1` vs `dx_cat_2`).

Expected input workbook layout (per pair workbook):
- Row 0: sparse category labels (exactly 2 categories, forward-filled)
- Row 1: exemplar diseases for each category
- Rows >= 2: clinical findings in column 0

This notebook is designed to run after the pair-input build step
(`20_differential_build_inputs.ipynb`).


In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — manifest-driven workbook selection + run modes
# ──────────────────────────────────────────────────────────────────────────────
# MANIFEST_PATH is the primary control surface for batch runs.
# It points to a CSV produced by build_differential_inputs that maps each pair input
# workbook to its intended output workbook and scenario metadata.
MANIFEST_PATH = PATHS.processed / "lr_differential" / "manifests" / "pairs_manifest.csv"
SCENARIO_FILTER = None   # e.g. ["chest_pain_carter_8", "metrorrhagia_sperle_tier3"]
MAX_PAIRS = None         # e.g. 20 for chunked runs

# Output location for model-specific differential-LR results.
# Expected layout: <OUTPUTS_BY_MODEL_ROOT>/<MODEL_ID>/<scenario_id>/*_filled.xlsx
OUTPUTS_BY_MODEL_ROOT = PATHS.processed / "lr_differential" / "outputs_by_model"

# Repair mode settings:
# - REPAIR_MODE=False: full run over manifest-selected workbooks.
# - REPAIR_MODE=True: patch only rows listed in INVALID_ROWS_PATH for MODEL_ID.
REPAIR_MODE = False
INVALID_ROWS_PATH = PATHS.processed / "lr_differential" / "manifests" / "invalid_rows.csv"
REPAIR_SCENARIO_FILTER = None  # e.g. ["chest_pain_carter_18"]
REPAIR_MAX_ROWS = None         # e.g. 25 for bounded repair batch

# Fallback manual mode (used only if MANIFEST_PATH is missing):
# Keep these paths in permanent data locations (data/raw + data/processed).
WORKBOOKS = [
    (
        PATHS.raw / "lr_matrices" / "dysphagia" / "LRs for 87 features within GI ddx - dysphagia vs esophageal pain without dysphagia.xlsx",
        PATHS.processed / "lr_differential" / "outputs" / "dysphagia" / "LRs for 87 features within GI ddx - dysphagia vs esophageal pain without dysphagia_filled.xlsx",
    )
]


## Run Behavior + Model-Switch Guide

### What this notebook does
- Selects workbook pairs from `MANIFEST_PATH` (or `WORKBOOKS` fallback)
- For each workbook, loads the two diagnosis categories + exemplars
- Estimates differential LR for **every non-empty finding row** (`row >= 2`)
- Writes model-specific outputs to `data/processed/lr_differential/outputs_by_model/<MODEL_ID>/...`

### Important run behavior
- Default (`REPAIR_MODE=False`): full recompute for selected workbook pairs.
- Repair mode (`REPAIR_MODE=True`): read `INVALID_ROWS_PATH` and recompute only listed rows for `MODEL_ID`.
- Invalid LR outputs (`<= 0`, NaN, or non-numeric) trigger one retry with stricter output instructions, then hard-fail.

### Model compatibility (for this notebook)
Use exact model IDs available to your OpenAI project (check your Models page). The table below is a practical compatibility guide for this notebook's current API calls.

| Family / Example `MODEL_ID` | Expected compatibility with current notebook | Recommended `API_BACKEND` | Extra settings to adjust | Notes |
|---|---|---|---|---|
| `gpt-4.1-nano-2025-04-14` | Yes | `responses` | Usually none | Lowest-cost default for smoke tests |
| `gpt-4.1-mini-2025-04-14` | Yes | `responses` | Usually none | Good quality/cost balance |
| `gpt-4.1-2025-04-14` | Yes | `responses` | Usually none | Strong non-reasoning baseline |
| `gpt-5-nano` / `gpt-5-mini` / `gpt-5` | Yes (if enabled in your org) | `responses` | Set `REASONING_EFFORT` (`minimal` or `low` to control cost) | `VERBOSITY` may be model-dependent |
| `o3` / `o4-mini` (if used) | Usually yes, but verify per account | `responses` or `chat` | Keep `REASONING_EFFORT`; set `API_BACKEND` explicitly if one path fails | Reasoning settings vary by model release |
| Preview / pro / research variants | Not guaranteed | Start with `responses` | May require notebook code changes if structured output behavior differs | Test on a small subset first |

### If switching models, what must be changed?
At minimum, change `MODEL_ID` in the config code cell.
Depending on model family, also check:
- `API_BACKEND`: `responses` (recommended), `chat`, or `auto`
- `REASONING_EFFORT`: most relevant for reasoning-capable models
- `VERBOSITY`: optional; unsupported values are ignored by fallback logic

Practical default for trying a new model:
1. set `MODEL_ID`
2. keep `API_BACKEND = "responses"`
3. set `REASONING_EFFORT = "minimal"` for lowest reasoning cost
4. keep `VERBOSITY = "medium"` unless docs/account behavior suggest otherwise

### Troubleshooting when switching models
- If parse calls fail on `responses`, try `API_BACKEND = "chat"`.
- If both backends fail, the model may not support this notebook's structured-output path.
- Always validate with a small run first (`MAX_PAIRS=1`) before full execution.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Differential-LR estimation — unified runner (Responses + Chat backends)
# ─────────────────────────────────────────────────────────────────────────────
# Design notes:
# - One code path for old/new model families.
# - Backend selected with API_BACKEND.
# - Optional fields (reasoning, verbosity) are attempted with safe fallbacks.
# - Full-run mode recomputes all finding rows; repair mode patches only invalid rows.
# ─────────────────────────────────────────────────────────────────────────────

CANONICAL_OUTPUTS_ROOT = PATHS.processed / "lr_differential" / "outputs"
MAX_RETRIES_PER_FINDING = 1


def generate_diff_gen_prompt(
    dx_cat_1: str,
    dx_cat_2: str,
    dx_cat_1_examples: list[str],
    dx_cat_2_examples: list[str],
) -> str:
    """Builds a concise, structured prompt for differential LR estimation."""

    def fmt_examples(ex_list: list[str]) -> str:
        return ", ".join(ex_list) if ex_list else "-"

    ex1 = fmt_examples(dx_cat_1_examples)
    ex2 = fmt_examples(dx_cat_2_examples)

    return f"""
You are a clinical epidemiologist.

Task: Estimate the *differential* likelihood ratio (LR) for a single clinical finding
that discriminates **{dx_cat_1}** from **{dx_cat_2}** in an outpatient diagnostic workup.
Assume the patient has exactly one of these two conditions.

Category examples:
• {dx_cat_1}: {ex1}
• {dx_cat_2}: {ex2}

Definition
LR = P(finding | {dx_cat_1}) / P(finding | {dx_cat_2})

Reference bands (for interpreting strength):
>10 strong for {dx_cat_1}; 5-10 moderate; 2-5 weak; 0.5-2 negligible;
0.2-0.5 weak for {dx_cat_2}; 0.1-0.2 moderate; <=0.1 strong for {dx_cat_2}.
""".strip()


def load_dx_categories(path: str | Path):
    """Extract exactly two categories and their exemplar lists from row 0/1."""
    df_raw = pd.read_excel(Path(path), header=None)
    cats = df_raw.iloc[0].ffill()
    ex_row = df_raw.iloc[1]

    unique_cats = pd.unique(cats.dropna())
    if len(unique_cats) != 2:
        raise ValueError("Expected exactly two category labels in row 0.")

    dx_cat_1, dx_cat_2 = unique_cats.tolist()
    ex1 = ex_row[cats == dx_cat_1].dropna().astype(str).tolist()
    ex2 = ex_row[cats == dx_cat_2].dropna().astype(str).tolist()
    return dx_cat_1, dx_cat_2, ex1, ex2


class DiffLR(BaseModel):
    lr: float = Field(gt=0)
    strength: str | None = None
    rationale: str | None = None


# Model/API controls:
# - MODEL_ID is the first knob to change when switching models.
# - API_BACKEND can be responses, chat, or auto.
# - REASONING_EFFORT and VERBOSITY are optional and model-dependent.
MODEL_ID = "gpt-4.1-nano-2025-04-14"
API_BACKEND = "responses"    # responses | chat | auto
REASONING_EFFORT = "low"     # minimal | low | medium | high
VERBOSITY = "medium"         # optional where supported

client = OpenAI()


def _dedupe_dicts(dicts: list[dict]) -> list[dict]:
    """Deduplicate kwargs candidates, including nested dict values."""
    seen = set()
    out = []
    for item in dicts:
        key = json.dumps(item, sort_keys=True)
        if key not in seen:
            seen.add(key)
            out.append(item)
    return out


def _validate_positive_lr(value: object) -> float:
    """Validate that parsed LR is finite and strictly positive."""
    try:
        lr_val = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"LR is not numeric: {value!r}") from exc

    if not math.isfinite(lr_val):
        raise ValueError(f"LR must be finite, got {lr_val!r}")
    if lr_val <= 0:
        raise ValueError(f"LR must be > 0, got {lr_val}")
    return lr_val


def _responses_parse_diff_lr(prompt: str, finding: str) -> float:
    """Try Responses API with optional fields, backing off if unsupported."""
    extra_candidates = [
        {
            **({"reasoning": {"effort": REASONING_EFFORT}} if REASONING_EFFORT else {}),
            **({"text": {"verbosity": VERBOSITY}} if VERBOSITY else {}),
        },
        {
            **({"reasoning": {"effort": REASONING_EFFORT}} if REASONING_EFFORT else {}),
        },
        {
            **({"text": {"verbosity": VERBOSITY}} if VERBOSITY else {}),
        },
        {},
    ]

    last_exc = None
    for extra in _dedupe_dicts(extra_candidates):
        try:
            resp = client.responses.parse(
                model=MODEL_ID,
                input=[
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": f"Finding: {finding}"},
                ],
                text_format=DiffLR,
                **extra,
            )
            return _validate_positive_lr(resp.output_parsed.lr)
        except Exception as exc:
            last_exc = exc

    raise RuntimeError(f"Responses API parse failed after fallbacks: {last_exc}")


def _chat_parse_diff_lr(prompt: str, finding: str) -> float:
    """Chat Completions structured parse fallback path."""
    extra = {}
    if REASONING_EFFORT and MODEL_ID.startswith("o"):
        extra["reasoning_effort"] = REASONING_EFFORT

    resp = client.beta.chat.completions.parse(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": f"Finding: {finding}"},
        ],
        response_format=DiffLR,
        **extra,
    )
    return _validate_positive_lr(resp.choices[0].message.parsed.lr)


def _estimate_once(prompt: str, finding: str) -> float:
    backend = API_BACKEND.lower().strip()
    if backend == "responses":
        return _responses_parse_diff_lr(prompt, finding)
    if backend == "chat":
        return _chat_parse_diff_lr(prompt, finding)
    if backend == "auto":
        try:
            return _responses_parse_diff_lr(prompt, finding)
        except Exception:
            return _chat_parse_diff_lr(prompt, finding)
    raise ValueError(f"Unsupported API_BACKEND: {API_BACKEND}")


def estimate_diff_lr(
    finding: str,
    dx_cat_1: str,
    dx_cat_2: str,
    ex1: list[str],
    ex2: list[str],
) -> float:
    """Estimate differential LR with one retry and strict failure semantics."""
    base_prompt = generate_diff_gen_prompt(dx_cat_1, dx_cat_2, ex1, ex2)
    failures: list[str] = []

    for attempt in range(MAX_RETRIES_PER_FINDING + 1):
        prompt = base_prompt
        if attempt > 0:
            prompt = (
                f"{base_prompt}\n\n"
                "Output constraint: return structured output with a single numeric field `lr`, "
                "and ensure lr is a finite float strictly greater than 0."
            )
        try:
            return _estimate_once(prompt, finding)
        except Exception as exc:
            failures.append(f"attempt={attempt + 1}: {exc}")

    failure_summary = " | ".join(failures)
    raise RuntimeError(
        f"Failed to estimate a valid differential LR after {MAX_RETRIES_PER_FINDING + 1} "
        f"attempt(s): {failure_summary}"
    )


def _normalize_scenario_filter(value):
    if value is None:
        return None
    if isinstance(value, str):
        return {value}
    return {str(item) for item in value}


def _resolve_model_output_path(manifest_output_path: Path, scenario_id: str) -> Path:
    """Map canonical manifest output path into model-scoped output directory."""
    try:
        rel = manifest_output_path.relative_to(CANONICAL_OUTPUTS_ROOT)
    except ValueError:
        rel = Path(scenario_id) / manifest_output_path.name
    return OUTPUTS_BY_MODEL_ROOT / MODEL_ID / rel


def _collect_finding_rows(df_in: pd.DataFrame) -> list[int]:
    rows: list[int] = []
    for row_idx in range(2, df_in.shape[0]):
        finding = str(df_in.iat[row_idx, 0]).strip()
        if finding:
            rows.append(row_idx)
    return rows


def load_workbooks_from_manifest(
    manifest_path: str | Path,
    scenario_filter=None,
    max_pairs: int | None = None,
):
    """Load input/output workbook records from manifest CSV."""
    manifest = Path(manifest_path)
    if not manifest.exists():
        raise FileNotFoundError(f"Manifest not found: {manifest}")

    df_manifest = pd.read_csv(manifest)
    required = {"input_workbook", "output_workbook"}
    missing = sorted(required - set(df_manifest.columns))
    if missing:
        raise ValueError(f"Manifest missing required columns: {missing}")

    if "scenario_id" not in df_manifest.columns:
        df_manifest["scenario_id"] = "unknown"
    if "pair_index" not in df_manifest.columns:
        df_manifest["pair_index"] = range(1, len(df_manifest) + 1)

    selected = df_manifest.sort_values(["scenario_id", "pair_index"]).copy()

    allowed = _normalize_scenario_filter(scenario_filter)
    if allowed is not None:
        selected = selected[selected["scenario_id"].isin(allowed)]

    if max_pairs is not None:
        selected = selected.head(int(max_pairs))

    records = []
    for row in selected.itertuples(index=False):
        records.append(
            {
                "input_workbook": REPO_ROOT / str(row.input_workbook),
                "manifest_output_workbook": REPO_ROOT / str(row.output_workbook),
                "scenario_id": str(row.scenario_id),
                "pair_index": int(row.pair_index),
            }
        )
    return records


def load_invalid_rows_for_repair(
    invalid_rows_path: str | Path,
    *,
    model_id: str,
    scenario_filter=None,
    max_rows: int | None = None,
) -> pd.DataFrame:
    """Load invalid rows emitted by audit script, scoped to the active model."""
    path = Path(invalid_rows_path)
    if not path.exists():
        raise FileNotFoundError(f"Invalid-rows CSV not found: {path}")

    df_invalid = pd.read_csv(path)
    required = {"model_id", "scenario_id", "pair_index", "input_workbook", "output_workbook", "row_index"}
    missing = sorted(required - set(df_invalid.columns))
    if missing:
        raise ValueError(f"Invalid-rows CSV missing required columns: {missing}")

    selected = df_invalid[df_invalid["model_id"] == model_id].copy()

    allowed = _normalize_scenario_filter(scenario_filter)
    if allowed is not None:
        selected = selected[selected["scenario_id"].isin(allowed)]

    selected = selected.sort_values(["scenario_id", "pair_index", "row_index"])

    if max_rows is not None:
        selected = selected.head(int(max_rows))

    return selected


def run_full_estimation(workbook_records: list[dict]) -> None:
    """Recompute all findings for each selected workbook pair."""
    if not workbook_records:
        raise ValueError("No workbooks selected. Check MANIFEST_PATH, SCENARIO_FILTER, and MAX_PAIRS.")

    print(f"Selected {len(workbook_records)} workbook pair(s) for full estimation.")

    for record in workbook_records:
        wb_in = Path(record["input_workbook"])
        wb_manifest_out = Path(record["manifest_output_workbook"])
        scenario_id = str(record["scenario_id"])
        pair_index = int(record["pair_index"])
        wb_out = _resolve_model_output_path(wb_manifest_out, scenario_id)

        print(f"\n{'=' * 80}")
        print(f"Scenario: {scenario_id} | Pair: {pair_index}")
        print(f"Input:  {wb_in}")
        print(f"Output: {wb_out}")
        print(f"{'=' * 80}")

        if not wb_in.exists():
            raise FileNotFoundError(f"Missing input workbook: {wb_in}")

        dx_cat_1, dx_cat_2, ex1, ex2 = load_dx_categories(wb_in)

        # Full-recompute behavior: always start from the raw input workbook.
        df_in = pd.read_excel(wb_in, sheet_name=0, header=None)
        df_out = df_in.copy()

        result_col = df_out.shape[1]
        df_out[result_col] = pd.Series([None] * len(df_out), dtype="object")

        header_label = f"Differential LR ({MODEL_ID} for '{dx_cat_1}' vs. '{dx_cat_2}')"
        df_out.iat[0, result_col] = header_label
        if len(df_out) > 1:
            df_out.iat[1, result_col] = ""

        finding_rows = _collect_finding_rows(df_in)
        print(f"Rows with findings: {len(finding_rows)}")

        for idx, row_idx in enumerate(finding_rows, start=1):
            finding = str(df_in.iat[row_idx, 0]).strip()
            print(
                f"[{idx}/{len(finding_rows)}] "
                f"scenario={scenario_id} | comparator={dx_cat_1} vs {dx_cat_2} | "
                f"finding={finding[:80]}"
            )

            try:
                lr_val = estimate_diff_lr(
                    finding=finding,
                    dx_cat_1=dx_cat_1,
                    dx_cat_2=dx_cat_2,
                    ex1=ex1,
                    ex2=ex2,
                )
            except Exception as exc:
                raise RuntimeError(
                    "Row estimation failed after retry: "
                    f"scenario={scenario_id}, pair={pair_index}, row={row_idx}, finding={finding!r}"
                ) from exc

            df_out.iat[row_idx, result_col] = lr_val

        wb_out.parent.mkdir(parents=True, exist_ok=True)
        df_out.to_excel(wb_out, index=False, header=False)
        print(f"Saved -> {wb_out}")


def run_repair_mode(invalid_rows_df: pd.DataFrame) -> None:
    """Patch only invalid rows listed by audit_differential_outputs.py."""
    if invalid_rows_df.empty:
        raise ValueError(f"No invalid rows found for model {MODEL_ID!r}.")

    print(f"Repair rows selected: {len(invalid_rows_df)} for model={MODEL_ID}")

    grouped = invalid_rows_df.groupby(["scenario_id", "input_workbook", "output_workbook"])
    for (scenario_id, input_workbook, output_workbook), group in grouped:
        wb_in = REPO_ROOT / str(input_workbook)
        wb_out = REPO_ROOT / str(output_workbook)

        print(f"\n{'=' * 80}")
        print(f"Repair scenario: {scenario_id}")
        print(f"Input:  {wb_in}")
        print(f"Output: {wb_out}")
        print(f"Rows to patch: {len(group)}")
        print(f"{'=' * 80}")

        if not wb_in.exists():
            raise FileNotFoundError(f"Missing input workbook for repair: {wb_in}")
        if not wb_out.exists():
            raise FileNotFoundError(f"Missing output workbook for repair: {wb_out}")

        dx_cat_1, dx_cat_2, ex1, ex2 = load_dx_categories(wb_in)
        df_in = pd.read_excel(wb_in, sheet_name=0, header=None)
        df_out = pd.read_excel(wb_out, sheet_name=0, header=None)
        result_col = df_out.shape[1] - 1

        ordered = group.sort_values(["pair_index", "row_index"])
        for idx, row in enumerate(ordered.itertuples(index=False), start=1):
            row_idx = int(row.row_index)
            if row_idx < 2 or row_idx >= df_in.shape[0]:
                raise IndexError(
                    f"Repair row_index out of range for {wb_in.name}: row_index={row_idx}"
                )

            finding = str(df_in.iat[row_idx, 0]).strip()
            if not finding:
                raise ValueError(
                    f"Repair row has empty finding text: scenario={scenario_id}, row_index={row_idx}"
                )

            print(
                f"[repair {idx}/{len(ordered)}] "
                f"scenario={scenario_id} | comparator={dx_cat_1} vs {dx_cat_2} | "
                f"finding={finding[:80]}"
            )

            try:
                lr_val = estimate_diff_lr(
                    finding=finding,
                    dx_cat_1=dx_cat_1,
                    dx_cat_2=dx_cat_2,
                    ex1=ex1,
                    ex2=ex2,
                )
            except Exception as exc:
                raise RuntimeError(
                    "Repair failed after retry: "
                    f"scenario={scenario_id}, row={row_idx}, finding={finding!r}"
                ) from exc

            df_out.iat[row_idx, result_col] = lr_val

        wb_out.parent.mkdir(parents=True, exist_ok=True)
        df_out.to_excel(wb_out, index=False, header=False)
        print(f"Repaired -> {wb_out}")


if REPAIR_MODE:
    invalid_rows_df = load_invalid_rows_for_repair(
        INVALID_ROWS_PATH,
        model_id=MODEL_ID,
        scenario_filter=REPAIR_SCENARIO_FILTER,
        max_rows=REPAIR_MAX_ROWS,
    )
    run_repair_mode(invalid_rows_df)
else:
    if Path(MANIFEST_PATH).exists():
        workbook_records = load_workbooks_from_manifest(
            MANIFEST_PATH,
            scenario_filter=SCENARIO_FILTER,
            max_pairs=MAX_PAIRS,
        )
    else:
        workbook_records = []
        for idx, (wb_in, wb_out) in enumerate(WORKBOOKS, start=1):
            workbook_records.append(
                {
                    "input_workbook": Path(wb_in),
                    "manifest_output_workbook": Path(wb_out),
                    "scenario_id": "manual",
                    "pair_index": idx,
                }
            )

    run_full_estimation(workbook_records)
